# Lectura Streaming con Campos de Observabilidad en PySpark

# Iniciar SparkSession y Definir el Esquema con Campos Observables

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, BooleanType

spark = (
    SparkSession.builder
    .appName("criptoanalytics-s09-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## Esquema del Contrato crypto.trade

In [2]:
esquema_crypto = (
    StructType()
    .add("tipoEvento", StringType())
    .add("tradeId", LongType())
    .add("symbol", StringType())
    .add("price", DoubleType())
    .add("quantity", DoubleType())
    .add("quoteQuantity", DoubleType())
    .add("isBuyerMaker", BooleanType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "binance-btc-trades")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str")
    )
    .select("topic", "partition", "offset", "kafkaTimestamp", from_json(col("json_str"), esquema_crypto).alias("e"))
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

## Campos de Observabilidad

In [3]:
observables = (
    eventos
    .withColumn("isValid",
        col("tipoEvento").isNotNull() & col("tradeId").isNotNull() &
        col("symbol").isNotNull() & col("timestamp").isNotNull() &
        (col("price") > 0) & (col("quantity") > 0) & (col("quoteQuantity") > 0))
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)

observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- tradeId: long (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- quoteQuantity: double (nullable = true)
 |-- isBuyerMaker: boolean (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



# Salida en Consola e Inspección de lastProgress (Throughput y Lag)

In [4]:
consulta_console = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

## Esperar unos micro-lotes e inspeccionar métricas

In [5]:
import time, json
time.sleep(12)

progreso = consulta_console.lastProgress
if progreso:
    metrics = progreso.get("sources", [{}])[0].get("metrics", {})
    log_observabilidad = {
        "service": "spark-streaming-cripto",
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metrics.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metrics.get("maxOffsetsBehindLatest")
    }
    print(json.dumps(log_observabilidad, indent=2))

{
  "service": "spark-streaming-cripto",
  "batchId": 2,
  "numInputRows": 416,
  "inputRowsPerSecond": 83.15010993403958,
  "processedRowsPerSecond": 1569.811320754717,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0"
}


# Persistencia y Análisis de Latencia Agregada por Ventana de Tiempo

In [6]:
PATH_PARQUET = "/home/jovyan/work/artifacts/btc_trades_observabilidad"
PATH_CHECKPOINT = "/home/jovyan/work/artifacts/chk_btc_observabilidad"

consulta_parquet = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", PATH_PARQUET)
    .option("checkpointLocation", PATH_CHECKPOINT)
    .trigger(processingTime="5 seconds")
    .start()
)

## Dejar persistir datos por 30 segundos

In [7]:
time.sleep(30)

consulta_console.stop()
consulta_parquet.stop()

## Lectura Batch de latencias agregadas por minuto

In [8]:
from pyspark.sql.functions import (window, avg, count, percentile_approx,
                                   min as spark_min, max as spark_max, col)

g = spark.read.parquet(PATH_PARQUET).filter("isValid")

lat = (g.groupBy(window("kafkaTimestamp", "1 minute"))
  .agg(count("*").alias("numTrades"),
       avg("latencyMs").alias("avgLatencyMs"),
       spark_min("latencyMs").alias("minLatencyMs"),
       percentile_approx("latencyMs", 0.95).alias("p95LatencyMs"),
       spark_max("latencyMs").alias("maxLatencyMs"))
  .select(col("window.start").alias("inicioVentana"), "numTrades",
          "avgLatencyMs", "minLatencyMs", "p95LatencyMs", "maxLatencyMs")
  .orderBy("inicioVentana"))
lat.show(30, truncate=False)

lat.agg(avg("avgLatencyMs").alias("avg_de_avg"),
        spark_max("avgLatencyMs").alias("max_de_avg"),
        spark_max("p95LatencyMs").alias("max_p95"),
        spark_max("maxLatencyMs").alias("max_abs")).show()

+-------------------+---------+------------------+------------+------------+------------+
|inicioVentana      |numTrades|avgLatencyMs      |minLatencyMs|p95LatencyMs|maxLatencyMs|
+-------------------+---------+------------------+------------+------------+------------+
|2026-10-15 00:25:00|437      |3554.3844393592676|86          |4836        |5035        |
+-------------------+---------+------------------+------------+------------+------------+

+------------------+------------------+-------+-------+
|        avg_de_avg|        max_de_avg|max_p95|max_abs|
+------------------+------------------+-------+-------+
|3554.3844393592676|3554.3844393592676|   4836|   5035|
+------------------+------------------+-------+-------+



In [9]:
from pyspark.sql.functions import window, avg, count

# 1. Leer la evidencia observable guardada en Parquet
path_parquet = "/home/jovyan/work/artifacts/btc_trades_observabilidad"
guardado = spark.read.parquet(path_parquet)

# 2. Calcular la latencia promedio agrupada por ventanas de 1 minuto
latencia_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(
        count("*").alias("numTrades"),
        avg("latencyMs").alias("avgLatencyMs")
    )
    .select(
        col("window.start").alias("inicioVentana"),
        col("window.end").alias("finVentana"),
        "numTrades",
        "avgLatencyMs"
    )
)

# 3. Filtrar aquellas ventanas que superen el umbral crítico de 3000 ms (3 segundos)
ventanas_criticas = latencia_minuto.filter(col("avgLatencyMs") > 3000)

# 4. Disparar la verificación lógica de la alerta
conteo_alertas = ventanas_criticas.count()

if conteo_alertas > 0:
    print(f"[ALERTA CRÍTICA] ¡Se encontraron {conteo_alertas} ventanas con latencia superior a 3000 ms!")
    ventanas_criticas.show(truncate=False)
else:
    print("[ESTADO SANO] El pipeline opera bajo el umbral de latencia crítica (ninguna ventana supera los 3000 ms).")


[ALERTA CRÍTICA] ¡Se encontraron 1 ventanas con latencia superior a 3000 ms!
+-------------------+-------------------+---------+------------------+
|inicioVentana      |finVentana         |numTrades|avgLatencyMs      |
+-------------------+-------------------+---------+------------------+
|2026-10-15 00:25:00|2026-10-15 00:26:00|437      |3554.3844393592676|
+-------------------+-------------------+---------+------------------+



In [10]:
from pyspark.sql.functions import min as smin, max as smax
r = g.agg(count("*").alias("n"), smin("kafkaTimestamp").alias("t0"),
          smax("kafkaTimestamp").alias("t1")).first()
print("trades/s promedio:", r.n / (r.t1 - r.t0).total_seconds())

trades/s promedio: 20.638518938320583
